
# ImageBind Multimodal Embedding Pipeline

This notebook generates multimodal embeddings for TikTok videos using ImageBind.

**Modalities**
- Text embeddings from post captions
- Audio embeddings from extracted audio tracks
- Vision embeddings from three representative video frames

The final output is a single dataset containing one embedding vector per modality for each video.


## 1. Imports and Model Initialization

In [ ]:

import os
import re
import subprocess
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict

import numpy as np
import pandas as pd
import torch

from imagebind import data
from imagebind.models.imagebind_model import imagebind_huge, ModalityType

device = "cuda" if torch.cuda.is_available() else "cpu"

model = imagebind_huge(pretrained=True)
model.eval()
model.to(device)


## 2. Load Dataset

In [ ]:

df = pd.read_csv('final_combined.csv')
df.head()


## 3. Text Cleaning

In [ ]:

URL_RE = re.compile(r"https?://\S+|www\.\S+")
HASHTAG_RE = re.compile(r"#\w+")

def clean_text(text):
    if pd.isna(text):
        return ""
    text = URL_RE.sub("", str(text))
    text = HASHTAG_RE.sub("", text)
    return text.strip()

df["body_clean"] = df["body"].apply(clean_text)


## 4. Audio Extraction

In [ ]:

def mp4_to_wav(mp4_path):
    wav_path = mp4_path.replace(".mp4", ".wav")

    if not os.path.exists(mp4_path):
        return None

    if os.path.exists(wav_path):
        return wav_path

    subprocess.run([
        "ffmpeg",
        "-i", mp4_path,
        "-vn",
        "-acodec", "pcm_s16le",
        "-ar", "44100",
        "-ac", "2",
        wav_path
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    return wav_path if os.path.exists(wav_path) else None

with ThreadPoolExecutor(max_workers=4) as executor:
    df["audio_path"] = list(executor.map(mp4_to_wav, df["video_path"]))


## 5. Generate Text Embeddings

In [ ]:

text_list = list(df["body_clean"])

text_input = data.load_and_transform_text(text_list, device)

with torch.no_grad():
    text_emb = model({
        ModalityType.TEXT: text_input
    })[ModalityType.TEXT]

text_emb = text_emb.cpu().numpy()

np.save("text_embeddings.npy", text_emb)
np.save("ids.npy", df["id"].values)


## 6. Generate Audio Embeddings

In [ ]:

def chunked(seq, batch_size):
    for i in range(0, len(seq), batch_size):
        yield seq[i:i + batch_size]

audio_paths = df["audio_path"].tolist()
audio_ids = df["id"].values

all_audio_embs = []
good_ids = []

with torch.no_grad():
    for batch_rows in chunked(list(zip(audio_ids, audio_paths)), 8):

        batch_ids = [x[0] for x in batch_rows]
        batch_paths = [x[1] for x in batch_rows]

        valid = [
            (vid, path)
            for vid, path in zip(batch_ids, batch_paths)
            if isinstance(path, str) and os.path.exists(path)
        ]

        if not valid:
            continue

        ids_batch, paths_batch = zip(*valid)

        audio_input = data.load_and_transform_audio_data(
            list(paths_batch),
            device
        )

        emb = model({
            ModalityType.AUDIO: audio_input
        })[ModalityType.AUDIO]

        all_audio_embs.append(emb.cpu().numpy())
        good_ids.extend(ids_batch)

audio_emb = np.concatenate(all_audio_embs)

np.save("audio_embeddings.npy", audio_emb)
np.save("audio_ids.npy", np.array(good_ids))


## 7. Generate Vision Embeddings

In [ ]:

# Use the final frame extraction functions developed during experimentation.

# Step 1:
# df['frames'] = df['video_path'].map(extract_three_frames)

# Step 2:
# Create ImageBind embeddings for each extracted frame.

# Step 3:
# Average the three frame embeddings to obtain one
# video-level embedding per TikTok.

# Save outputs:
# np.save('vision_embeddings.npy', video_embs)
# np.save('vision_ids.npy', video_ids)


## 8. Merge All Modalities

In [ ]:

text_emb = np.load("text_embeddings.npy")
ids = np.load("ids.npy")

audio_emb = np.load("audio_embeddings.npy")
audio_ids = np.load("audio_ids.npy")

video_embs = np.load("vision_embeddings.npy")
video_ids = np.load("vision_ids.npy")

df_text = pd.DataFrame({
    "id": ids,
    "text_embedding": list(text_emb)
})

df_audio = pd.DataFrame({
    "id": audio_ids,
    "audio_embedding": list(audio_emb)
})

df_video = pd.DataFrame({
    "id": video_ids,
    "video_embedding": list(video_embs)
})

df_emb = (
    df_text
    .merge(df_audio, on="id", how="left")
    .merge(df_video, on="id", how="left")
)

df_emb.to_csv("tiktok_id_with_embeddings.csv", index=False)

df_emb.head()
